In [1]:
import pandas as pd
import math
import nbformat
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px

In [2]:
url = (
    "https://media.githubusercontent.com/media/"
    "StackExchange/Survey/refs/heads/main/"
    "packages/archive/2025/results.csv"
)

df = pd.read_csv(url, low_memory=False)

In [3]:
# amostra 5 linhas
df.head()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0


In [4]:
print(f"Total de registros: {len(df):,}")
print(f"Total de variáveis: {df.shape[1]:,}")
print(f"Respondentes únicos {df['ResponseId'].nunique():,}")
print(f"IDs duplicados: {df["ResponseId"].duplicated().sum():,}")
print(f"IDs ausentes: {df['ResponseId'].isna().sum():,}")

Total de registros: 49,191
Total de variáveis: 172
Respondentes únicos 49,191
IDs duplicados: 0
IDs ausentes: 0


## 1. Definição da população

**Fonte:** Stack Overflow Developer Survey 2025.

**População de interesse:** respondentes registrados na base pública da pesquisa de 2025.

**Unidade amostral:** cada respondente individual, identificado pela variável `ResponseId`.

**Dimensões da base:**
- 49.191 observações;
- 172 variáveis;
- 49.191 identificadores únicos;
- Nenhum identificador duplicado ou ausente.

A análise será realizada considerando os respondentes da pesquisa, sem pressupor representatividade estatística de todos os desenvolvedores do mundo.

In [5]:
# info sobre as variáveis
colunas = [
    "Age",
    "WorkExp",
    "AISelect",
    "Country",
    "Employment"
]

df[colunas].info()

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Age         49191 non-null  str    
 1   WorkExp     42893 non-null  float64
 2   AISelect    33720 non-null  str    
 3   Country     35437 non-null  str    
 4   Employment  48339 non-null  str    
dtypes: float64(1), str(4)
memory usage: 1.9 MB


In [6]:
# verificação de dados ausentes
df[colunas].isna().sum()

Age               0
WorkExp        6298
AISelect      15471
Country       13754
Employment      852
dtype: int64

In [7]:
# entendendo o uso da IA
df["AISelect"].value_counts(dropna = False)

AISelect
Yes, I use AI tools daily                      15883
NaN                                            15471
Yes, I use AI tools weekly                      5958
No, and I don't plan to                         5454
Yes, I use AI tools monthly or infrequently     4628
No, but I plan to soon                          1797
Name: count, dtype: int64

In [8]:
numerador = df['AISelect'].str.startswith("Yes").sum()
denominador = df['AISelect'].count()

proporcao_ia = numerador / denominador

print(f"Proporção: {proporcao_ia:.4f}")
print(f"Porcentagem: {proporcao_ia:.2%}")


Proporção: 0.7850
Porcentagem: 78.50%


In [9]:
print(f"Porcentagem de respondentes que preencheram a pergunta AISelect afirmam utilizar ferramentas de IA: {proporcao_ia:.2%}")

Porcentagem de respondentes que preencheram a pergunta AISelect afirmam utilizar ferramentas de IA: 78.50%


## 2. Dimensionamento e seleção da amostra

**População inicial:** 49.191 respondentes.

**Método de amostragem:** Amostragem Aleatória Simples (AAS), sem reposição.

**Parâmetros propostos:**
- Nível de confiança: 95%;
- Margem de erro: 5 pontos percentuais;
- Proporção esperada: 50%.

O objetivo é dimensionar uma amostra capaz de estimar proporções populacionais com a precisão desejada, sob as condições estatísticas estabelecidas.

In [10]:
# tamanho da nossa população
N = 49191
# valor critico z
z = 1.96
# proporção esperada da nossa hipotese (50% da nossa base observada utiliza IA)
p = 0.50
# intervalo da margem de erro 5%
E = 0.05

In [11]:
# Dimensionamento inicial da amostra
n0 = (z**2 * p * (1 - p)) / E**2

print(f"Tamanho amostral inicial: {n0:.2f}")

Tamanho amostral inicial: 384.16


In [12]:
# Correção para população finita
n = (N * n0) / (N + n0 - 1)

print(f"Tamanho amostral corrigido: {n:.2f}")

Tamanho amostral corrigido: 381.19


In [13]:
# Arredondamento para cima
n_final = math.ceil(n)

print(f"Tamanho final da amostra: {n_final}")

Tamanho final da amostra: 382


In [14]:
# Amostragem aleatória simples (AAS)
amostra = df.sample(
    n=n_final,
    replace=False,
    random_state=42
)

In [15]:
amostra["AISelect"].value_counts(dropna=False)

AISelect
NaN                                            122
Yes, I use AI tools daily                      122
Yes, I use AI tools monthly or infrequently     45
Yes, I use AI tools weekly                      42
No, and I don't plan to                         41
No, but I plan to soon                          10
Name: count, dtype: int64

In [16]:
# Proporção de respostas válidas em AISelect
taxa_validas = df["AISelect"].count() / len(df)

print(f"Taxa de respostas válidas: {taxa_validas:.2%}")

Taxa de respostas válidas: 68.55%


In [17]:
# Ajuste pela taxa de respostas válidas
n_ajustado = n_final / taxa_validas

# Arredondamento para cima
n_ajustado = math.ceil(n_ajustado)

print(f"Tamanho ajustado: {n_ajustado}")

Tamanho ajustado: 558


In [18]:
amostra_ajustada = df.sample(
    n=n_ajustado,
    replace=False,
    random_state=42
)

In [19]:
print(f"Respondentes sorteados: {len(amostra_ajustada)}")

print(
    f"Respondentes únicos: "
    f"{amostra_ajustada['ResponseId'].nunique()}"
)

print(
    f"Respostas válidas em AISelect: "
    f"{amostra_ajustada['AISelect'].count()}"
)

print(
    f"Respostas ausentes em AISelect: "
    f"{amostra_ajustada['AISelect'].isna().sum()}"
)

Respondentes sorteados: 558
Respondentes únicos: 558
Respostas válidas em AISelect: 374
Respostas ausentes em AISelect: 184


In [20]:
numerador_amostra = amostra_ajustada["AISelect"].str.startswith("Yes").sum()
denominador_amostra = amostra_ajustada["AISelect"].count()

proporcao_ia = numerador_amostra / denominador_amostra

In [21]:
print(f"A porcentagem de pessoas que utilizam IA na amostra é de {proporcao_ia:.2%}")

A porcentagem de pessoas que utilizam IA na amostra é de 81.82%


In [22]:
colunas = ["Age", "WorkExp", "AISelect", "Country", "Employment"]

df[colunas].describe(include="all")

,Age,WorkExp,AISelect,Country,Employment
count,49191,42893.000000,33720,35437,48339
unique,7,NaN,5,177,6
top,25-34 years old,NaN,"Yes, I use AI tools daily",United States of America,Employed
freq,16519,NaN,15883,7233,33750
mean,NaN,13.367403,NaN,NaN,NaN
std,NaN,10.800117,NaN,NaN,NaN
min,NaN,1.000000,NaN,NaN,NaN
25%,NaN,5.000000,NaN,NaN,NaN
50%,NaN,10.000000,NaN,NaN,NaN
75%,NaN,20.000000,NaN,NaN,NaN


In [23]:
fig = px.histogram(
    df,
    x="WorkExp",
    nbins=30,
    title="Distribuição dos anos de experiência profissional",
    labels={"WorkExp": "Anos de experiência"},
    template="plotly_dark"
)

fig.update_layout(
    yaxis_title="Número de respondentes",
    bargap=0.05
)

fig.show()

In [24]:
resumo_exp = df["WorkExp"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max"
])

display(resumo_exp.round(2))

count     42893.00
mean         13.37
median       10.00
std          10.80
min           1.00
max         100.00
Name: WorkExp, dtype: float64

In [25]:
distribuicao_ia = (
    df["AISelect"]
    .fillna("Sem resposta")
    .value_counts()
    .reset_index()
)

fig = px.bar(
    distribuicao_ia,
    x="count",
    y="AISelect",
    orientation="h",
    title="Distribuição do uso de IA",
    labels={
        "count": "Número de respondentes",
        "AISelect": "Frequência de uso"
    },
    template="plotly_dark"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [26]:
def resumo_estatistico(base):
    return pd.Series({
        "Respondentes": len(base),
        "Respostas válidas IA": base["AISelect"].count(),
        "Uso de IA (%)": base["AISelect"].str.startswith("Yes").mean() * 100,
        "Experiência média": base["WorkExp"].mean(),
        "Experiência mediana": base["WorkExp"].median(),
        "Desvio-padrão": base["WorkExp"].std()
    })

resumo = pd.DataFrame({
    "População": resumo_estatistico(df),
    "Amostra": resumo_estatistico(amostra_ajustada)
})

display(resumo.round(2))

,População,Amostra
Respondentes,49191.00,558.00
Respostas válidas IA,33720.00,374.00
Uso de IA (%),53.81,54.84
Experiência média,13.37,14.31
Experiência mediana,10.00,11.00
Desvio-padrão,10.80,12.36


In [27]:
import plotly.express as px

ordem_idades = [
    "Under 18 years old",
    "18-24 years old",
    "25-34 years old",
    "35-44 years old",
    "45-54 years old",
    "55-64 years old",
    "65 years or older",
    "Prefer not to say"
]

def grafico_idades(base, titulo):
    dados = (
        base["Age"]
        .value_counts(normalize=True)
        .mul(100)
        .rename_axis("Faixa etária")
        .reset_index(name="Percentual")
    )

    fig = px.bar(
        dados,
        x="Faixa etária",
        y="Percentual",
        category_orders={"Faixa etária": ordem_idades},
        title=titulo,
        template="plotly_dark"
    )

    fig.update_layout(xaxis_tickangle=-35)
    fig.show()

grafico_idades(df, "Faixas etárias — População")
grafico_idades(amostra_ajustada, "Faixas etárias — Amostra")

In [29]:
dados_exp = pd.concat([
    df[["WorkExp"]].assign(Grupo="População"),
    amostra_ajustada[["WorkExp"]].assign(Grupo="Amostra")
])

fig = px.histogram(
    dados_exp,
    x="WorkExp",
    color="Grupo",
    histnorm="percent",
    barmode="overlay",
    nbins=30,
    opacity=0.65,
    title="Distribuição de experiência — População vs. Amostra",
    labels={"WorkExp": "Anos de experiência"},
    template="plotly_dark"
)

fig.show()

In [30]:
def distribuicao_ia(base, grupo):
    dados = (
        base["AISelect"]
        .fillna("Sem resposta")
        .value_counts(normalize=True)
        .mul(100)
        .rename_axis("Resposta")
        .reset_index(name="Percentual")
    )

    dados["Grupo"] = grupo
    return dados

dados_ia = pd.concat([
    distribuicao_ia(df, "População"),
    distribuicao_ia(amostra_ajustada, "Amostra")
])

fig = px.bar(
    dados_ia,
    x="Percentual",
    y="Resposta",
    color="Grupo",
    barmode="group",
    orientation="h",
    title="Uso de IA — População vs. Amostra",
    template="plotly_dark"
)

fig.show()